In [16]:
import pandas as pd
df = pd.read_csv("listing_flagged.csv")
print(df.shape)

(566673, 71)


In [17]:
flag_cols = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_bedrooms_flag",
    "invalid_bathrooms_flag",
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag"
]

df[flag_cols].sum().sort_values(ascending=False)

missing_geo_flag               80441
invalid_living_area_flag         376
out_of_ca_flag                   301
negative_timeline_flag           277
purchase_after_close_flag        266
invalid_longitude_flag            86
listing_after_close_flag          74
zero_geo_flag                     65
invalid_days_on_market_flag       29
invalid_close_price_flag           0
invalid_bedrooms_flag              0
invalid_bathrooms_flag             0
dtype: int64

In [18]:
# Convert numeric columns
numeric_cols = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\nNumeric conversion complete")
print(df[numeric_cols].dtypes)


Numeric conversion complete
ClosePrice      float64
LivingArea      float64
DaysOnMarket      int64
dtype: object


In [19]:
# Read existing business-rule flags
# These flags were already created in previous weeks

rule_flags = [
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_numeric_flag"
]

for col in rule_flags:

    if col in df.columns:

        # Convert to Boolean
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
        )
        print(f"{col} loaded successfully")

    else:
        print(f"Warning: {col} not found")


invalid_close_price_flag loaded successfully
invalid_living_area_flag loaded successfully
invalid_days_on_market_flag loaded successfully


In [20]:
#  Define IQR outlier function

def add_iqr_flag(df, column_name):

    # Calculate quartiles
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)

   
    IQR = Q3 - Q1

  
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

   
    flag_col = f"{column_name}_iqr_outlier_flag"
 # Flag outliers
    df[flag_col] = (
        (df[column_name] < lower) |
        (df[column_name] > upper)
    )

    # Print IQR summary
    print("\n===================================")
    print(f"{column_name} IQR Summary")
    print("===================================")

    print(f"Q1: {Q1}")
    print(f"Q3: {Q3}")
    print(f"IQR: {IQR}")

    print(f"Lower Bound: {lower}")
    print(f"Upper Bound: {upper}")

    print(
        f"Outliers Flagged: {df[flag_col].sum()}"
    )

    return df

In [21]:
# Apply IQR outlier detection
# Apply IQR filtering to:
# ClosePrice
# LivingArea
# DaysOnMarket

df = add_iqr_flag(df, "ClosePrice")

df = add_iqr_flag(df, "LivingArea")

df = add_iqr_flag(df, "DaysOnMarket")


ClosePrice IQR Summary
Q1: 600000.0
Q3: 1350000.0
IQR: 750000.0
Lower Bound: -525000.0
Upper Bound: 2475000.0
Outliers Flagged: 10668

LivingArea IQR Summary
Q1: 1248.0
Q3: 2301.0
IQR: 1053.0
Lower Bound: -331.5
Upper Bound: 3880.5
Outliers Flagged: 28079

DaysOnMarket IQR Summary
Q1: 5.0
Q3: 22.0
IQR: 17.0
Lower Bound: -20.5
Upper Bound: 47.5
Outliers Flagged: 49253


In [22]:
# Create combined IQR outlier flag
# If any numeric field is an outlier,
# mark the row as True

df["any_iqr_outlier_flag"] = (

    df["ClosePrice_iqr_outlier_flag"] |

    df["LivingArea_iqr_outlier_flag"] |

    df["DaysOnMarket_iqr_outlier_flag"]

)

print("\nCombined IQR flag created")

print(
    df["any_iqr_outlier_flag"]
    .value_counts()
)



Combined IQR flag created
any_iqr_outlier_flag
False    485820
True      80853
Name: count, dtype: int64


In [23]:
# Create final exclusion flag- Combine:
# rule invalid records + IQR outliers
if "invalid_numeric_flag" in df.columns:

    df["exclude_from_analysis_flag"] = (

        df["invalid_numeric_flag"] |

        df["any_iqr_outlier_flag"]

    )

else:

    df["exclude_from_analysis_flag"] = (
        df["any_iqr_outlier_flag"]
    )

print("\nFinal exclusion flag created")

print(
    df["exclude_from_analysis_flag"]
    .value_counts()
)


Final exclusion flag created
exclude_from_analysis_flag
False    485820
True      80853
Name: count, dtype: int64


In [24]:
#  Create clean filtered dataset
# Keep only records
# that are NOT excluded

df_filtered = df[
    ~df["exclude_from_analysis_flag"]
].copy()

print("\nFiltered dataset created")

print("Original rows:", len(df))

print("Filtered rows:", len(df_filtered))


Filtered dataset created
Original rows: 566673
Filtered rows: 485820


In [ ]:
# Compare before and after filtering

comparison_rows = []

fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in fields:

    outlier_flag_col = f"{col}_iqr_outlier_flag"

    comparison_rows.append({

        "Field": col,

        "Median Before": df[col].median(),

        "Median After": df_filtered[col].median(),

        "Mean Before": df[col].mean(),

        "Mean After": df_filtered[col].mean(),


    })

comparison = pd.DataFrame(comparison_rows)
print("Before vs After Filtering Comparison")

print(comparison)

Before vs After Filtering Comparison
          Field  Median Before  Median After   Mean Before     Mean After
0    ClosePrice       855000.0      828500.0  1.205656e+06  950011.598336
1    LivingArea         1670.0        1611.0  1.980580e+03    1742.108336
2  DaysOnMarket           11.0          10.0  1.910784e+01      12.580415
